# Pipe Network Solver

A steady-state 1D model of a network of pipes. Given fixed pressures at some nodes (such as a supply tank) and the flow drawn off at others, it calculates the pressure at every node and the flow in every pipe.

## Physics
- **Pressure drop:** Darcy–Weisbach, $\Delta P = f \cdot \frac{L}{D} \cdot \frac{\rho v^2}{2}$
- **Friction factor:** $64/Re$ for laminar flow; the Colebrook equation for turbulent flow, solved by root-finding (`scipy.optimize.brentq`)
- **Network equations:** one momentum equation per pipe and one mass balance per unknown node, solved together with `scipy.optimize.fsolve`

In [8]:
import numpy as np
from scipy.optimize import brentq, fsolve

# Physical constants for water at room temp
RHO = 998.0  # kg/m3
MU = 1.002e-3  # Pa s

def colebrook(f, Re, e_D):
    """Colebrook equation function for root-finding."""
    if f <= 0:
        return 1.0
    return 1.0 / np.sqrt(f) + 2.0 * np.log10(e_D / 3.7 + 2.51 / (Re * np.sqrt(f)))

def get_friction_factor(Re, e_D):
    """Calculates friction factor using Brent's method for turbulent flow."""
    if Re < 2300:
        return 64.0 / Re if Re > 0 else 0.0
    else:
        # Removed silent fallback; raising Brent root finder error if it fails.
        # Widen search interval from 0.1 to 1.0 to handle high relative roughness/low Re cases.
        f_solved = brentq(colebrook, 1e-5, 1.0, args=(Re, e_D))
        return f_solved

def pipe_pressure_drop(flow_rate, L, D, roughness):
    """Calculates pressure drop (Pa) for a given flow rate (m3/s) in a pipe."""
    if flow_rate == 0:
        return 0.0

    area = np.pi * (D ** 2) / 4.0
    velocity = abs(flow_rate) / area
    Re = RHO * velocity * D / MU
    e_D = roughness / D

    f = get_friction_factor(Re, e_D)
    dp = f * (L / D) * RHO * (velocity ** 2) / 2.0

    return np.sign(flow_rate) * dp

In [3]:
# Single Pipe Validation (Water, 5 L/s, 100 m of 50 mm steel pipe)
L_test = 100.0
D_test = 0.050
roughness_steel = 4.5e-5  # 0.045 mm
flow_test = 0.005  # 5 L/s = 0.005 m3/s

dp = pipe_pressure_drop(flow_test, L_test, D_test, roughness_steel)
print(f"Calculated pressure drop: {dp / 1e5:.3f} bar (Expected: ~1.39 bar)")

Calculated pressure drop: 1.382 bar (Expected: ~1.39 bar)


In [4]:
class PipeNetwork:
    def __init__(self, nodes, pipes):
        """
        nodes: dict of node_id: {pressure_fixed: value or None, external_flow: value}
        pipes: list of dicts {from_node, to_node, L, D, roughness}
        """
        self.nodes = nodes
        self.pipes = pipes
        self.node_ids = list(nodes.keys())
        self.num_nodes = len(self.node_ids)
        self.num_pipes = len(pipes)

        # Map nodes to indices
        self.node_idx = {nid: i for i, nid in enumerate(self.node_ids)}

    def solve(self):
        # Variables to solve: Q for each pipe (size: num_pipes), P for each node (size: num_nodes)
        # Total variables = num_pipes + num_nodes
        num_vars = self.num_pipes + self.num_nodes

        def residual(x):
            Q = x[:self.num_pipes]
            P = x[self.num_pipes:]
            res = np.zeros(num_vars)

            # 1. Momentum equations for each pipe: P_from - P_to - delta_P = 0
            for i, pipe in enumerate(self.pipes):
                idx_from = self.node_idx[pipe['from_node']]
                idx_to = self.node_idx[pipe['to_node']]
                dp = pipe_pressure_drop(Q[i], pipe['L'], pipe['D'], pipe['roughness'])
                res[i] = P[idx_from] - P[idx_to] - dp

            # 2. Node equations
            for idx, nid in enumerate(self.node_ids):
                node = self.nodes[nid]
                res_idx = self.num_pipes + idx

                if node.get('pressure_fixed') is not None:
                    # Node pressure is fixed
                    res[res_idx] = P[idx] - node['pressure_fixed']
                else:
                    # Mass balance at the node: Sum of Q_in - Sum of Q_out = external_flow_drawn
                    flow_sum = 0.0
                    for p_idx, pipe in enumerate(self.pipes):
                        if pipe['to_node'] == nid:
                            flow_sum += Q[p_idx]
                        elif pipe['from_node'] == nid:
                            flow_sum -= Q[p_idx]
                    res[res_idx] = flow_sum - node.get('external_flow', 0.0)

            return res

        # Initial guess
        initial_guess = np.zeros(num_vars)
        # Guess atmospheric pressure for nodes and small positive flow for pipes
        initial_guess[:self.num_pipes] = 0.001
        initial_guess[self.num_pipes:] = 101325.0

        # Fix guessed pressure for defined source nodes
        for idx, nid in enumerate(self.node_ids):
            if self.nodes[nid].get('pressure_fixed') is not None:
                initial_guess[self.num_pipes + idx] = self.nodes[nid]['pressure_fixed']

        solution, info, ier, mesg = fsolve(residual, initial_guess, full_output=True)
        if ier != 1:
            raise RuntimeError(f"Solver failed to converge: {mesg}")

        solved_Q = solution[:self.num_pipes]
        solved_P = solution[self.num_pipes:]

        return {
            'flows': {f"Pipe {i+1} ({p['from_node']} -> {p['to_node']})": q for i, (p, q) in enumerate(zip(self.pipes, solved_Q))},
            'pressures': {nid: p for nid, p in zip(self.node_ids, solved_P)},
            'raw_flows': solved_Q,
            'raw_pressures': solved_P
        }

In [5]:
# Validate Parallel Pipes splitting flow 50/50
nodes_parallel = {
    'A': {'pressure_fixed': 200000.0, 'external_flow': 0.0},  # Inlet at 2 bar
    'B': {'pressure_fixed': 100000.0, 'external_flow': 0.0}   # Outlet at 1 bar
}
pipes_parallel = [
    {'from_node': 'A', 'to_node': 'B', 'L': 100.0, 'D': 0.05, 'roughness': 1e-5},
    {'from_node': 'A', 'to_node': 'B', 'L': 100.0, 'D': 0.05, 'roughness': 1e-5}  # Identical parallel pipe
]

network = PipeNetwork(nodes_parallel, pipes_parallel)
results = network.solve()

print("Parallel Pipes Test Results:")
for pipe, flow in results['flows'].items():
    print(f"  {pipe}: {flow * 1000:.3f} L/s")

Parallel Pipes Test Results:
  Pipe 1 (A -> B): 4.563 L/s
  Pipe 2 (A -> B): 4.563 L/s


In [6]:
# 1. Laminar Flow Validation (Matches Hagen-Poiseuille)
# Hagen-Poiseuille: dP = 128 * MU * L * Q / (pi * D^4)
L_lam = 50.0
D_lam = 0.01
Q_lam = 1.0e-6  # extremely low flow rate for guaranteed laminar flow (Re ~ 127)

# Analytical pressure drop:
dp_analytical = 128.0 * MU * L_lam * Q_lam / (np.pi * D_lam**4)

# Calculated pressure drop:
dp_calc = pipe_pressure_drop(Q_lam, L_lam, D_lam, roughness=0.0)

print("Laminar Flow Test:")
print(f"  Hagen-Poiseuille dP: {dp_analytical:.4f} Pa")
print(f"  Solver calculated dP: {dp_calc:.4f} Pa")
print(f"  Relative Error: {abs(dp_calc - dp_analytical) / dp_analytical * 100:.6f}%")

Laminar Flow Test:
  Hagen-Poiseuille dP: 204.1258 Pa
  Solver calculated dP: 204.1258 Pa
  Relative Error: 0.000000%


In [7]:
# 2. Mass Balance & Pipe Direction Validation
# Node A -> Node B -> Node C
# Node B has a demand (external_flow) of 1 L/s (0.001 m3/s)
# Node C has fixed pressure (0 Pa, gauge)
# We'll also test pipe reversal by defining Pipe 2 from C to B (backwards flow should be negative)
nodes_net = {
    'A': {'pressure_fixed': 150000.0, 'external_flow': 0.0},
    'B': {'pressure_fixed': None, 'external_flow': 0.001},
    'C': {'pressure_fixed': 100000.0, 'external_flow': 0.0}
}
pipes_net = [
    {'from_node': 'A', 'to_node': 'B', 'L': 50.0, 'D': 0.05, 'roughness': 1.5e-5},
    {'from_node': 'C', 'to_node': 'B', 'L': 50.0, 'D': 0.05, 'roughness': 1.5e-5}  # Reversed direction
]

net = PipeNetwork(nodes_net, pipes_net)
res = net.solve()

print("Mass Balance & Direction Test Results:")
for pipe, flow in res['flows'].items():
    print(f"  {pipe}: {flow * 1000:.3f} L/s")
print(f"  Intermediate Node B Pressure: {res['pressures']['B'] / 1e5:.4f} bar")

# Mass balance check: Q_AB (in to B) + Q_CB (in to B) = External flow drawn (1 L/s)
q_in = res['raw_flows'][0] + res['raw_flows'][1]
print(f"  Total net flow into Node B: {q_in * 1000:.3f} L/s (Expected: 1.000 L/s)")

Mass Balance & Direction Test Results:
  Pipe 1 (A -> B): 3.554 L/s
  Pipe 2 (C -> B): -2.554 L/s
  Intermediate Node B Pressure: 1.1765 bar
  Total net flow into Node B: 1.000 L/s (Expected: 1.000 L/s)


In [9]:
# 3. Looped Ring Network Simulation
# Topology:
# - Node 'Tank' (Fixed supply pressure at 3.0 bar)
# - Node 'Outlet_1' (Draws off 8.0 L/s)
# - Node 'Outlet_2' (Draws off 5.0 L/s)
#
# Pipes form a triangle loop:
# - Pipe 1: Tank -> Outlet_1 (80 m, 50 mm ID)
# - Pipe 2: Tank -> Outlet_2 (100 m, 50 mm ID)
# - Pipe 3: Outlet_1 -> Outlet_2 (50 m, 40 mm ID) [The connecting loop pipe]

nodes_loop = {
    'Tank': {'pressure_fixed': 300000.0, 'external_flow': 0.0},
    'Outlet_1': {'pressure_fixed': None, 'external_flow': 0.008}, # 8 L/s draw-off
    'Outlet_2': {'pressure_fixed': None, 'external_flow': 0.005}  # 5 L/s draw-off
}

pipes_loop = [
    {'from_node': 'Tank', 'to_node': 'Outlet_1', 'L': 80.0, 'D': 0.050, 'roughness': 2.0e-5},
    {'from_node': 'Tank', 'to_node': 'Outlet_2', 'L': 100.0, 'D': 0.050, 'roughness': 2.0e-5},
    {'from_node': 'Outlet_1', 'to_node': 'Outlet_2', 'L': 50.0, 'D': 0.040, 'roughness': 2.0e-5}
]

loop_network = PipeNetwork(nodes_loop, pipes_loop)
loop_results = loop_network.solve()

print("Looped Network Results:")
print("-----------------------")
print("Flow Rates:")
for pipe_name, flow_rate in loop_results['flows'].items():
    print(f"  {pipe_name}: {flow_rate * 1000:.3f} L/s")

print("\nNode Pressures (Gauge):")
for node_id, press in loop_results['pressures'].items():
    print(f"  Node {node_id}: {press / 1e5:.4f} bar")

# Quick manual validation of Loop closure pressure balance:
# dp1 - dp2 - dp3 should sum close to zero around the loop depending on sign directions
print("\nMass Balance Check:")
total_demand = sum(n['external_flow'] for n in nodes_loop.values())
total_source = loop_results['raw_flows'][0] + loop_results['raw_flows'][1]
print(f"  Total Source Flow (Tank): {total_source * 1000:.3f} L/s")
print(f"  Total System Demand: {total_demand * 1000:.3f} L/s")

Looped Network Results:
-----------------------
Flow Rates:
  Pipe 1 (Tank -> Outlet_1): 6.979 L/s
  Pipe 2 (Tank -> Outlet_2): 6.021 L/s
  Pipe 3 (Outlet_1 -> Outlet_2): -1.021 L/s

Node Pressures (Gauge):
  Node Tank: 3.0000 bar
  Node Outlet_1: 1.1310 bar
  Node Outlet_2: 1.2315 bar

Mass Balance Check:
  Total Source Flow (Tank): 13.000 L/s
  Total System Demand: 13.000 L/s
